In [ ]:
#importing packages and env-var

from pyspark.sql.session import SparkSession
import pyspark.sql.functions as f

import os
from dotenv import load_dotenv

env = load_dotenv(dotenv_path='/mnt/d/linux/project/vars/.env')

#env vars for postgresql connection
mssqlsource = os.getenv('MSSQLSOURCE')
mssqluser = os.getenv('MSSQL_SPARK_USER')
mssqlpwd = os.getenv('MSSQL_SPARK_PWD')
mssqldriver = os.getenv('MSSQL_DRIVER')
sqlserverjar = os.getenv('SQLSERVER_JAR')
mssql_dwdest = os.getenv('MSSQL_CONN')

In [ ]:
#creating sparksession 

spark = ((
    
        SparkSession
            .builder
            .appName('dimpackage')
            .config('spark.jars', f'{sqlserverjar}')
            .config('spark.driver.extraClassPath', f'{sqlserverjar}')
            .config('spark.executor.extraClassPath', f'{sqlserverjar}')
        .getOrCreate()
))

In [ ]:
#load data from table into a dataframe
dimpackage_frame = ((spark
                     .read
                     .format('jdbc')
                     .option('url', mssqlsource)
                     .option('user', mssqluser)
                     .option('password', mssqlpwd)
                     .option('driver', mssqldriver)
                     .option('query',
                            """
                                select

                                    packagetypeid,
                                    packagetypename
                                
                                from warehouse.packagetypes
                             
                            """)
                )).load()

In [ ]:
#rename and reshape data type
renameframe = ((
    
                dimpackage_frame
                    .withColumnRenamed('packagetypeid', 'packageid')
                    .withColumnRenamed('packagetypename', 'packagetype')


))

In [ ]:
#load transformed data into a database table

((
    renameframe
        .write
        .format('jdbc')
        .option('url', mssql_dwdest)
        .option('user', mssqluser)
        .option('password', mssqlpwd)
        .option('driver', mssqldriver)
        .option('dbtable', 'dimension.packagetype')
        .option('mode', 'overwrite')
    .save()
    
))

In [ ]:
spark.stop()